In [21]:
!wget -O blaze_face_short_range.tflite \
https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite



--2026-01-05 00:43:32--  https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite
Resolving storage.googleapis.com (storage.googleapis.com)... 142.250.98.207, 173.194.216.207, 108.177.11.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|142.250.98.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 229746 (224K) [application/octet-stream]
Saving to: ‘blaze_face_short_range.tflite’

blaze_face_short_ra 100%[===================>] 224.36K  --.-KB/s    in 0.002s  

2026-01-05 00:43:33 (95.2 MB/s) - ‘blaze_face_short_range.tflite’ saved [229746/229746]



In [ ]:
import os
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import mediapipe as mp
import torchvision.models as models

from torch.utils.data import Dataset, DataLoader, random_split
from mediapipe.tasks import python
from mediapipe.tasks.python import vision



class DriverActionClassifier(nn.Module):
    def __init__(self, backbone, num_classes=10):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Linear(3 * 576, num_classes)

    def forward(self, image, face, hand):
        im = self.backbone(image).flatten(1)
        f = self.backbone(face).flatten(1)
        h = self.backbone(hand).flatten(1)
        x = torch.cat([im, f, h], dim=1)
        return self.classifier(x)


class DriverDataset(Dataset):
    def __init__(self, root_dir, detector, img_size=224):
        self.img_size = img_size
        self.detector = detector
        self.samples = []

        classes = sorted(
            c for c in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, c))
        )
        self.class_map = {c: i for i, c in enumerate(classes)}

        for c in classes:
            class_dir = os.path.join(root_dir, c)
            for f in os.listdir(class_dir):
                if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.samples.append((os.path.join(class_dir, f), self.class_map[c]))

    def __len__(self):
        return len(self.samples)

    def _preprocess(self, img):
        img = cv2.resize(img, (self.img_size, self.img_size))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = img.astype(np.float32) / 127.5 - 1.0
        return torch.from_numpy(img).permute(2, 0, 1)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        frame = cv2.imread(path)
        if frame is None:
            raise RuntimeError(f"Failed to read {path}")

        H, W, _ = frame.shape
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        result = self.detector.detect(mp_image)

        if result.detections:
            box = result.detections[0].bounding_box
            x1 = max(0, int(box.origin_x - 0.25 * box.width))
            y1 = max(0, int(box.origin_y - 0.25 * box.height))
            x2 = min(W, int(box.origin_x + box.width * 1.25))
            y2 = min(H, int(box.origin_y + box.height * 1.25))
            face = frame[y1:y2, x1:x2]
        else:
            face = frame.copy()

        hand = frame[H//2:H, W//2:W]
        full = frame.copy()

        return (
            self._preprocess(full),
            self._preprocess(face),
            self._preprocess(hand),
            label
        )


BaseOptions = mp.tasks.BaseOptions
FaceDetectorOptions = mp.tasks.vision.FaceDetectorOptions
VisionRunningMode = mp.tasks.vision.RunningMode

detector = mp.tasks.vision.FaceDetector.create_from_options(
    FaceDetectorOptions(
        base_options=BaseOptions(
            model_asset_path="/content/blaze_face_short_range.tflite.task"
        ),
        running_mode=VisionRunningMode.IMAGE,
        min_detection_confidence=0.5
    )
)


device = "cuda" if torch.cuda.is_available() else "cpu"

DATASET_ROOT = "/Users/rushilmohan/Google Drive/My Drive/Distracted Driver Detection/state-farm-distracted-driver-detection/imgs/train"  # CHANGE IF USING DRIVE

dataset = DriverDataset(DATASET_ROOT, detector)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2)


mobilenet = models.mobilenet_v3_small(weights="IMAGENET1K_V1")
mobilenet.classifier = nn.Identity()

model = DriverActionClassifier(mobilenet, num_classes=10).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4)


best_val_loss = float("inf")

for epoch in range(20):
    model.train()
    train_loss = 0

    for full, face, hand, labels in train_loader:
        full, face, hand, labels = (
            full.to(device),
            face.to(device),
            hand.to(device),
            labels.to(device)
        )

        logits = model(full, face, hand)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    model.eval()
    val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for full, face, hand, labels in val_loader:
            full, face, hand, labels = (
                full.to(device),
                face.to(device),
                hand.to(device),
                labels.to(device)
            )

            logits = model(full, face, hand)
            loss = criterion(logits, labels)

            val_loss += loss.item()
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_loss /= len(val_loader)
    val_acc = correct / total

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_driver_action_model.pth")

    print(
        f"Epoch {epoch+1:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )


FileNotFoundError: [Errno 2] No such file or directory: '/Users/rushilmohan/Google Drive/My Drive/Distracted Driver Detection/state-farm-distracted-driver-detection/imgs/train'